# PROJECT SUBMISSION BY - VIVA BARANWAL
# Comprehensive Marketing Campaign & Customer Acquisition Analysis
## Domain Context: The 4 Ps Marketing Mix
This project applies data science techniques to analyze customer acquisition data structured around the classical **Marketing Mix (Product, Price, Place, and Promotion)**. The analysis focuses deeply on understanding consumer profiles (**People**), checking behavior across distribution channels (**Place**), identifying high-value items (**Product**), and measuring the conversion rates of active marketing initiatives (**Promotion**).

### Objective
To execute rigorous Exploratory Data Analysis (EDA) and apply inferential statistical hypothesis testing to identify the core socioeconomic and structural factors driving consumer purchasing choices.


In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

# Load data and strip accidental whitespace from headers
df = pd.read_csv("data/marketing_data.csv")
df.columns = df.columns.str.strip()

## Step 1: Initial Data Audit and Data Type Validation
Before conducting any mathematical profiling, we check structural integrity. We specifically audit `Dt_Customer` and `Income` to verify if they have been accurately imported or require parsing from text strings into explicit numeric/datetime types.

In [3]:
print("--- Initial Structural Check ---")
print(df[['Income', 'Dt_Customer']].dtypes)
print("\nSample Data Snapshots:")
print(df[['Income', 'Dt_Customer']].head(3))

--- Initial Structural Check ---
Income         str
Dt_Customer    str
dtype: object

Sample Data Snapshots:
        Income Dt_Customer
0  $84,835.00      6/16/14
1  $57,091.00      6/15/14
2  $67,267.00      5/13/14


## Step 2: Categorical Standardization and Group-Based Income Imputation
Raw data contains irregular values within the `Education` and `Marital_Status` fields that require explicit group alignment. Once clean, we address missing values in the `Income` field by calculating the median income within a customer's specific socioeconomic peer group (matched by Education and Marital Status), which avoids the skewing effect of a global mean.

In [4]:
# Clean text markers
df['Income'] = df['Income'].str.replace('$', '', regex=False).str.replace(',', '', regex=False).str.strip()
df['Income'] = pd.to_numeric(df['Income'], errors='coerce')
df['Dt_Customer'] = pd.to_datetime(df['Dt_Customer'], format='%m/%d/%y')

# Standardize categories
marital_map = {'Married': 'Married', 'Together': 'Together', 'Divorced': 'Divorced', 'Widow': 'Widow', 'Single': 'Single', 'Alone': 'Single', 'Absurd': 'Single', 'YOLO': 'Single'}
df['Marital_Status'] = df['Marital_Status'].map(marital_map)

education_map = {'Graduation': 'Graduate', 'PhD': 'Postgraduate', 'Master': 'Postgraduate', '2n Cycle': 'Undergraduate', 'Basic': 'Undergraduate'}
df['Education'] = df['Education'].map(education_map)

# Group-based dynamic median imputation
df['Income'] = df.groupby(['Education', 'Marital_Status'])['Income'].transform(lambda x: x.fillna(x.median()))
print("Remaining Missing Values in Income:", df['Income'].isnull().sum())

Remaining Missing Values in Income: 0


## Step 3: Feature Engineering and Behavioral Metric Derivation
To build a better analytical framework, we derive new continuous indicators representing family demographics and purchasing power. We aggregate children markers (`Total_Children`), establish consumer `Age` relative to data tracking baselines, and compute cumulative transaction volume (`Total_Purchases`) alongside gross item sales (`Total_Spending`).

In [5]:
df['Total_Children'] = df['Kidhome'] + df['Teenhome']
df['Age'] = 2015 - df['Year_Birth']

product_cols = ['MntWines', 'MntFruits', 'MntMeatProducts', 'MntFishProducts', 'MntSweetProducts', 'MntGoldProds']
df['Total_Spending'] = df[product_cols].sum(axis=1)

purchase_cols = ['NumWebPurchases', 'NumCatalogPurchases', 'NumStorePurchases']
df['Total_Purchases'] = df[purchase_cols].sum(axis=1)

print(df[['Age', 'Total_Children', 'Total_Spending', 'Total_Purchases']].head(3))

   Age  Total_Children  Total_Spending  Total_Purchases
0   45               0            1190               14
1   54               0             577               17
2   57               1             251               10


## Step 4: Distribution Profiling and Outlier Treatment via IQR Capping
Using continuous numeric features like income and age without handling distribution extremes can introduce severe error biases in our downstream statistical modeling. We generate box plots and histograms to visualize the distributions and apply Interquartile Range (IQR) capping boundaries to handle anomalies safely.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.histplot(df['Income'], kde=True, ax=axes[0]).set(title='Income Distribution')
sns.boxplot(x=df['Age'], ax=axes[1]).set(title='Age Box Plot')
plt.show()

# Interquartile Range outlier boundary treatment
for col in ['Income', 'Age']:
    Q1, Q3 = df[col].quantile(0.25), df[col].quantile(0.75)
    IQR = Q3 - Q1
    df[col] = np.where(df[col] < (Q1 - 1.5 * IQR), Q1 - 1.5 * IQR, df[col])
    df[col] = np.where(df[col] > (Q3 + 1.5 * IQR), Q3 + 1.5 * IQR, df[col])
print("Outlier boundaries successfully treated.")

## Step 5: Advanced Categorical Encoding (Ordinal and One-Hot Transformations)
Machine learning and statistical models cannot process raw structural strings directly. We apply **Ordinal Encoding** to ordered metrics like `Education` to preserve the logical sequence of academic tiers, and **One-Hot Encoding** to non-ordinal categories like marital state and location to avoid implying numerical rank where none exists.

In [6]:
edu_order = {'Undergraduate': 0, 'Graduate': 1, 'Postgraduate': 2}
df['Education_Encoded'] = df['Education'].map(edu_order)

df_encoded = pd.get_dummies(df, columns=['Marital_Status', 'Country'], drop_first=True, dtype=int)
print("Encoded feature array dimensions:", df_encoded.shape)

Encoded feature array dimensions: (2240, 42)


## Step 6: Multivariate Association and Feature Correlation Mapping
We compute a Pearson correlation matrix and map it onto a visualization heatmap. This lets us systematically inspect collinear relationships between income metrics, physical shopping distributions, digital channel traffic, and overall purchase activities.

In [ ]:
core_cols = ['Age', 'Income', 'Total_Children', 'Total_Spending', 'Total_Purchases', 'NumWebPurchases', 'NumStorePurchases', 'NumCatalogPurchases', 'NumWebVisitsMonth']
plt.figure(figsize=(8, 6))
sns.heatmap(df_encoded[core_cols].corr(), annot=True, cmap='coolwarm', fmt='.2f', linewidths=0.5)
plt.title('Correlation Heatmap of Primary Numerical Matrix')
plt.show()

## Step 7: Hypothesis Testing and Statistical Inference
We translate core corporate behavior theories into structured statistical evaluations. We evaluate age-based store dependency, online adoption among time-constrained parents, channel-to-channel cannibalization vectors, and geographic volume variations against rigorous mathematical benchmarks.

In [ ]:
print("=== INFERENTIAL HYPOTHESIS TESTING INTERPRETATIONS ===\n")

# H1: Age vs Store Purchases (Pearson Correlation)
r_age, p_age = stats.pearsonr(df_encoded['Age'], df_encoded['NumStorePurchases'])
print(f"[Hypothesis 7a] Age/Store Association Coefficient: {r_age:.3f} (p-value: {p_age:.3e})")

# H2: Online Adoption for Families with Children (Two-Sample Welch's T-Test)
kids_web = df_encoded[df_encoded['Total_Children'] > 0]['NumWebPurchases']
no_kids_web = df_encoded[df_encoded['Total_Children'] == 0]['NumWebPurchases']
t_kids, p_kids = stats.ttest_ind(kids_web, no_kids_web, equal_var=False)
print(f"[Hypothesis 7b] Parental Web Volume Deviation T-Stat: {t_kids:.3f} (p-value: {p_kids:.3e})")

# H3: Sales Channel Cannibalization Vectors (Pearson Correlation Matrix)
r_web, _ = stats.pearsonr(df_encoded['NumStorePurchases'], df_encoded['NumWebPurchases'])
r_cat, _ = stats.pearsonr(df_encoded['NumStorePurchases'], df_encoded['NumCatalogPurchases'])
print(f"[Hypothesis 7c] Channel Interdependence -> Store vs Web: {r_web:.3f}, Store vs Catalog: {r_cat:.3f}")

# H4: US Purchase Output vs Rest of World Performance (Two-Sample Welch's T-Test)
us_col = [c for c in df_encoded.columns if 'US' in c]
if us_col:
    us_vol = df_encoded[df_encoded[us_col[0]] == 1]['Total_Purchases']
    row_vol = df_encoded[df_encoded[us_col[0]] == 0]['Total_Purchases']
    t_us, p_us = stats.ttest_ind(us_vol, row_vol, equal_var=False)
    print(f"[Hypothesis 7d] US Outperformance Metric T-Stat: {t_us:.3f} (p-value: {p_us:.3e})")

## Step 8: Visual Analytics and Business Intelligence Reporting
To make our analytical discoveries easily actionable for marketing decision-makers, we build data visualizations. This final reporting dashboard tracks item revenue performance, target age distributions across promotional runs, market geographical concentrations, and consumer friction vectors like complaints.

In [ ]:
fig, axes = plt.subplots(3, 2, figsize=(14, 18))

# 8a. Product Revenue Hierarchy
product_totals = df[product_cols].sum().sort_values(ascending=False)
sns.barplot(x=product_totals.values, y=product_totals.index, ax=axes[0, 0], hue=product_totals.index, legend=False, palette='viridis')
axes[0, 0].set_title('Product Revenue Hierarchy Optimization Profile')

# 8b. Age Distribution vs Last Campaign Acceptance
sns.regplot(x='Age', y='Response', data=df, logistic=True, ax=axes[0, 1], scatter_kws={'alpha':0.1}, line_kws={'color':'red'})
axes[0, 1].set_title('Age Density vs Final Campaign Acceptance Curve')

# 8c. Country Specific Campaign Acceptance Value
sns.countplot(x='Country', hue='Response', data=df, ax=axes[1, 0], palette='Set2')
axes[1, 0].set_title('Campaign Acceptance Volume per Geographic Region')

# 8d. Household Children Volume vs Expenditure Profile
sns.boxplot(x='Total_Children', y='Total_Spending', data=df, ax=axes[1, 1], palette='pastel')
axes[1, 1].set_title('Household Children Density vs Expenditure Distribution')

# 8e. Educational Profile of Complaining Customers
complainers = df[df['Complain'] == 1]
sns.countplot(x='Education', data=complainers, ax=axes[2, 0], hue='Education', legend=False, palette='rocket', order=['Undergraduate', 'Graduate', 'Postgraduate'])
axes[2, 0].set_title('Socioeconomic Friction: Complaints by Education Tier')

fig.delaxes(axes[2, 1])
plt.tight_layout()
plt.show()

## DONE
